# F1 Halo Removal via Video Inpainting
### CSCI 365 — Digital Image Processing, Spring 2026

**Goal:** Remove the Halo safety device from F1 onboard footage using a two-stage pipeline:
1. **Mask Detection** — per-frame Sobel-Y arch contour detection + tilt-aware keel probe fitting
2. **Neural Inpainting** — fill the masked region using two learned methods:
   - *Method 1:* LaMa Large-Mask Inpainting — Fast Fourier Convolution network with global receptive field
   - *Method 2:* RAFT Optical Flow Temporal Propagation — dense CNN flow estimation + backward warp

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings; warnings.filterwarnings('ignore')
%matplotlib inline

DATA = Path('data')
DATA.mkdir(exist_ok=True)
print('Setup complete. OpenCV', cv2.__version__, '| PyTorch', torch.__version__)


## 1. Data

Download the source video (720p, ~100 MB) and extract a working clip.

> **Lab machines:** if the video has been placed in the shared data space, set `VIDEO_PATH`
> to that path and skip the download cell.

In [ ]:
import yt_dlp
import subprocess, os

URL = 'https://www.youtube.com/watch?v=CPCu4n5LIeg'

existing = list(DATA.glob('suzuka_raw.*'))

def _probe_codec(p: Path):
    try:
        cmd = ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
               '-show_entries', 'stream=codec_name',
               '-of', 'default=noprint_wrappers=1:nokey=1', str(p)]
        res = subprocess.run(cmd, capture_output=True, text=True)
        return res.stdout.strip()
    except Exception:
        return None

# If there's an existing download, check its codec and re-download if it's AV1
if existing:
    existing = sorted(existing)
    cand = existing[0]
    codec = _probe_codec(cand)
    if codec == 'av1':
        print(f'Existing download {cand.name} uses AV1 — removing and re-downloading non-AV1 variant')
        try:
            cand.unlink()
        except Exception as e:
            print('Failed to remove existing file, will attempt to overwrite:', e)
        existing = []

if not existing:
    print('Downloading (720p), preferring MP4/H.264 non-AV1 stream...')
    opts = {
        # Prefer MP4 containers and non-AV1 video codec when available to avoid AV1 decode issues
        # If no non-AV1 MP4 is available, yt-dlp will fall back to the best available format.
        'format': 'bestvideo[ext=mp4][height<=720][vcodec!=av1]+bestaudio/best[ext=mp4][height<=720]',
        'outtmpl': str(DATA / 'suzuka_raw.%(ext)s'),
        'merge_output_format': 'mp4',
        'quiet': True,
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        ydl.download([URL])
    existing = list(DATA.glob('suzuka_raw.*'))

if not existing:
    raise RuntimeError('Download failed — no suzuka_raw.* file found in data/ after yt-dlp')

VIDEO_PATH = existing[0]
print('Video:', VIDEO_PATH.name, '  ', round(VIDEO_PATH.stat().st_size / 1e6, 1), 'MB')

In [ ]:
import subprocess

# Prefer a pre-created H.264 copy if present (suzuka_raw_h264.mp4), otherwise probe
# the downloaded file's codec and transcode if it's AV1. This avoids the repeated
# "Missing Sequence Header" AV1 messages on some platforms (WSL/OpenCV build).

# If the downloader produced a separate H.264 file earlier, prefer it
h264_alt = DATA / (VIDEO_PATH.stem + '_h264.mp4')
if h264_alt.exists():
    print('Using existing H.264 copy:', h264_alt.name)
    VIDEO_READ_PATH = h264_alt
else:
    codec = _probe_codec(VIDEO_PATH)
    if codec == 'av1':
        print('Input is AV1; transcoding to H.264 for reliable VideoCapture...')
        out = VIDEO_PATH.with_name(VIDEO_PATH.stem + '_h264.mp4')
        ffmpeg_cmd = [
            'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
            '-i', str(VIDEO_PATH),
            '-c:v', 'libx264', '-crf', '18', '-preset', 'fast',
            '-c:a', 'copy', str(out),
        ]
        subprocess.run(ffmpeg_cmd, check=True)
        VIDEO_READ_PATH = out
    else:
        VIDEO_READ_PATH = VIDEO_PATH

print('Opening video:', VIDEO_READ_PATH.name)
cap = cv2.VideoCapture(str(VIDEO_READ_PATH), cv2.CAP_FFMPEG)
FPS   = cap.get(cv2.CAP_PROP_FPS)
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W     = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(str(W) + 'x' + str(H), ' ', round(FPS, 1), 'fps', ' ', TOTAL, 'frames (',
      round(TOTAL / FPS, 1), 's )')

# ── Adjust these to choose your clip window ─────────────────────────────────
T_START  =  5.0   # seconds into the video
DURATION =  5.0   # 5s @ 60fps = 300 frames — keep short for fast iteration
# ─────────────────────────────────────────────────────────────────────────────

n0 = int(T_START * FPS)
nF = int(DURATION * FPS)

cap.set(cv2.CAP_PROP_POS_FRAMES, n0)
raw_bgr = []
for _ in range(nF):
    ok, frm = cap.read()
    if not ok:
        break
    raw_bgr.append(frm)
cap.release()

if len(raw_bgr) == 0:
    raise RuntimeError('No frames were read from the video. Check that ffmpeg/ffprobe are installed and that the file is readable.')

frames = np.array([cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in raw_bgr], dtype=np.uint8)
print('Clip:', len(frames), 'frames  shape', frames.shape)

## 2. Frame Exploration

Examine the clip to find the Halo strut's approximate position before building a mask.
Note the pixel coordinates — you will need them in the next section.

In [ ]:
picks6 = np.linspace(0, len(frames) - 1, 6, dtype=int)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, i in zip(axes.flat, picks6):
    ax.imshow(frames[i])
    ax.set_title('frame ' + str(i))
    ax.axis('off')
plt.suptitle('Sample frames — locate the Halo strut (dark vertical bar, centre of frame)',
             fontsize=13)
plt.tight_layout()
plt.show()

print('Frame size:', W, 'x', H, '(w x h)')
print('Approx centre of frame: x =', W // 2, ' y =', H // 2)

## 3. Halo Mask Creation

The rough mask is a simple **T-shape**: a full-width arch band (top ~28 %) + a narrow
centre keel column. This is the initial mask for frame 0.

Earlier attempts at dark-pixel refinement consistently made the mask *worse* — the
wide-angle lens produces corner vignetting that looks dark, so the refiner kept those
and dropped actual arch body pixels. The rough T-shape is more reliable: it definitly
covers all of the Halo at the cost of masking a little sky, which the inpainter fills
back in cleanly.

In [ ]:
# ── ARCH: full arch band ─────────────────────────────────────────────────────
ARCH_Y0 = 0.00
ARCH_Y1 = 0.28   # arch body extends ~28 % down; 15 % clips it and leaves lower arch visible
# ── STRUT: centre keel below the arch ────────────────────────────────────────
STRUT_HW = 0.04   # half-width each side
STRUT_Y0 = 0.25   # slight overlap with ARCH_Y1
STRUT_Y1 = 0.65
STRUT_SEARCH_X0 = 0.40   # slightly wider than 0.42 — keel can sit off-centre
STRUT_SEARCH_X1 = 0.60
# ─────────────────────────────────────────────────────────────────────────────

mask_arch = np.zeros((H, W), dtype=np.uint8)
mask_arch[int(ARCH_Y0*H):int(ARCH_Y1*H), :] = 255

def detect_strut_cx(frame_rgb, arch_y1, strut_y1, x0=0.40, x1=0.60):
    gray = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)
    fh, fw = gray.shape
    ry0 = int(arch_y1 * fh)
    ry1 = min(int((arch_y1 + 0.12) * fh), int(strut_y1 * fh))
    rx0, rx1 = int(x0 * fw), int(x1 * fw)
    roi = gray[ry0:ry1, rx0:rx1].astype(np.float32)
    col_mean   = roi.mean(axis=0)
    col_smooth = cv2.GaussianBlur(col_mean.reshape(1, -1), (21, 1), 0).ravel()
    cx_local   = int(np.argmin(col_smooth))
    return (rx0 + cx_local) / fw

STRUT_CX = detect_strut_cx(frames[0], ARCH_Y1, STRUT_Y1,
                            STRUT_SEARCH_X0, STRUT_SEARCH_X1)
print(f'Auto-detected strut centre: {STRUT_CX:.3f}  (x = {int(STRUT_CX * W)} px)')

sx0 = int((STRUT_CX - STRUT_HW) * W)
sx1 = int((STRUT_CX + STRUT_HW) * W)
mask_strut = np.zeros((H, W), dtype=np.uint8)
mask_strut[int(STRUT_Y0*H):int(STRUT_Y1*H), sx0:sx1] = 255

mask_rough = cv2.bitwise_or(mask_arch, mask_strut)

def overlay_mask(frame, mask, alpha=0.55, color=(255, 0, 0)):
    out = frame.astype(float).copy()
    out[mask.astype(bool)] = out[mask.astype(bool)] * (1 - alpha) + np.array(color) * alpha
    return out.astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(frames[0]);                              axes[0].set_title('Original');        axes[0].axis('off')
axes[1].imshow(overlay_mask(frames[0], mask_arch));     axes[1].set_title('Arch band');       axes[1].axis('off')
axes[2].imshow(overlay_mask(frames[0], mask_rough));    axes[2].set_title('Arch + strut (MASK)'); axes[2].axis('off')
plt.suptitle(f'T-shape mask — STRUT_CX={STRUT_CX:.3f} (x={int(STRUT_CX*W)}px)', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Dark-pixel refinement was discarded: wide-angle corner vignetting is dark, so the
# refiner kept lens artifacts and dropped arch body pixels. The T-shape rough mask
# covers the full arch reliably — any extra sky it includes gets cleanly filled.
MASK = mask_rough.copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(frames[0]);               axes[0].set_title('Original');  axes[0].axis('off')
axes[1].imshow(overlay_mask(frames[0], MASK)); axes[1].set_title('MASK (T-shape)'); axes[1].axis('off')
plt.suptitle('Initial mask — rough T-shape used directly (no dark-pixel refinement)', fontsize=11)
plt.tight_layout(); plt.show()

px = int(MASK.sum() / 255)
print('Mask covers', px, 'px  (', round(100 * px / (H * W), 2), '% of frame )')

## 4. Per-Frame Halo Mask Detection

**Why not optical flow / LK tracking?**

LK tracking accumulates drift. Tracking cockpit features works until the steering
wheel rotates and the hands sweep across the frame, injecting non-rigid motion into
the affine estimate.

**New approach — detect the arch independently in every frame:**

The Halo arch top bar is *always* fully visible: it sits above the driver's head and
is never occluded by hands or the steering wheel.  Its bottom edge — the transition
from dark titanium to bright sky — is a strong, sharp horizontal edge detectable with
Sobel-Y in each frame independently.

1. **Arch body mask** — apply Sobel-Y in the top band to estimate the arch lower
   edge, then fill a controlled body region above/below that edge without reaching
   the roof/visor strip.
2. **Keel mask** — re-detect the darkest centre column per frame.  A robust line fit
   rejects hand/wheel outliers and carries the last known centre through occlusion.
3. **Logo + cleanup** — extract the fixed F1 logo as a small white contour, then
   close/dilate the final mask slightly to cover sunlit Halo borders.

In [ ]:
# Mask detection tunables: keep these in one place for classroom-demo runs.
# Logo dimensions measured from frame 0: dark panel extends to x~224, y~84.
LOGO_W_PX = 230
LOGO_H_PX = 88
LOGO_CROP_W = 180
LOGO_CROP_H = 82
LOGO_MIN_AREA = 20

ARCH_Y_MAX = 0.255
ARCH_CONTOUR_PAD = 12
ARCH_WEAK_EDGE_FRAC = 0.12
ARCH_BODY_TO_TOP = True
ARCH_BODY_ABOVE = 92
ARCH_BODY_BELOW = 4
ARCH_BODY_CLOSE_KERNEL = (23, 7)

CONTROL_KEEL_TOP_HALF_WIDTH = 26
CONTROL_KEEL_BOTTOM_HALF_WIDTH = 12
CONTROL_KEEL_JUNCTION_BONUS = 8
CONTROL_KEEL_JUNCTION_ROWS = 34
CONTROL_YOKE_ROWS = 44
CONTROL_YOKE_TOP_HALF_WIDTH = 72
CONTROL_YOKE_BOTTOM_HALF_WIDTH = 30
CONTROL_JUNCTION_ELLIPSE_X = 62
CONTROL_JUNCTION_ELLIPSE_Y = 38
CONTROL_JUNCTION_ELLIPSE_DY = 18
CONTROL_FINAL_CLOSE_KERNEL = (9, 7)
CONTROL_HOLE_CLOSE_KERNEL = (21, 11)
CONTROL_FINAL_DILATE_KERNEL = (7, 5)
CONTROL_FINAL_DILATE_ITER = 1


KEEL_SEARCH_X0 = 0.42
KEEL_SEARCH_X1 = 0.70
KEEL_PROBE_YS = [175, 185, 195, 205, 215, 225, 240, 255, 270, 300]
KEEL_PROBE_HALF_H = 6
KEEL_MIN_CONTRAST = 18
KEEL_MAX_VALLEY_WIDTH = 90
KEEL_INLIER_RESID_PX = 18
KEEL_MAX_SLOPE = 1.30
KEEL_Y1 = 0.65
KEEL_TOP_HALF_WIDTH = 38
KEEL_BOTTOM_HALF_WIDTH = 18
KEEL_JUNCTION_BONUS = 14
KEEL_JUNCTION_ROWS = 44
KEEL_YOKE_ROWS = 78
KEEL_YOKE_TOP_HALF_WIDTH = 118
KEEL_YOKE_BOTTOM_HALF_WIDTH = KEEL_TOP_HALF_WIDTH + KEEL_JUNCTION_BONUS


def _fit_keel_line(ys, cxs, max_slope=KEEL_MAX_SLOPE, inlier_resid=KEEL_INLIER_RESID_PX):
    ys = np.asarray(ys, dtype=np.float64)
    cxs = np.asarray(cxs, dtype=np.float64)
    n = len(ys)
    best = None

    for i in range(n):
        for j in range(i + 1, n):
            dy = ys[j] - ys[i]
            if abs(dy) < 1:
                continue
            slope = (cxs[j] - cxs[i]) / dy
            if abs(slope) > max_slope:
                continue
            intercept = cxs[i] - slope * ys[i]
            residuals = np.abs(cxs - (slope * ys + intercept))
            keep = residuals <= inlier_resid
            score = (int(keep.sum()), -float(np.median(residuals[keep])) if keep.any() else -999.0, -abs(slope))
            if best is None or score > best[0]:
                best = (score, keep, slope, intercept)

    if best is None:
        return 0.0, float(np.median(cxs)), np.ones(n, dtype=bool)

    keep = best[1]
    if keep.sum() >= 2:
        slope, intercept = np.polyfit(ys[keep], cxs[keep], 1)
        if abs(slope) > max_slope:
            slope, intercept = 0.0, float(np.median(cxs[keep]))
        return float(slope), float(intercept), keep

    return float(best[2]), float(best[3]), keep


def _extract_f1_logo_mask(frame_rgb, search):
    """Extract the fixed white F1 broadcast logo as contours, not a rectangle."""
    h, w = search.shape[:2]
    cw = min(LOGO_CROP_W, w)
    ch = min(LOGO_CROP_H, h)
    crop = frame_rgb[:ch, :cw]
    hsv = cv2.cvtColor(crop, cv2.COLOR_RGB2HSV)
    white = ((hsv[:, :, 1] < 65) & (hsv[:, :, 2] > 160)).astype(np.uint8) * 255
    white = cv2.morphologyEx(white, cv2.MORPH_OPEN,
                             cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    white = cv2.morphologyEx(white, cv2.MORPH_CLOSE,
                             cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 3)))

    contours, _ = cv2.findContours(white, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    logo = np.zeros((h, w), dtype=np.uint8)
    kept = 0
    for cnt in contours:
        area = cv2.contourArea(cnt)
        x, y, bw, bh = cv2.boundingRect(cnt)
        if area < LOGO_MIN_AREA or x > 170 or y > 80 or bw > 130 or bh > 50:
            continue
        cv2.drawContours(logo[:ch, :cw], [cnt], -1, 255, thickness=cv2.FILLED)
        kept += 1
    logo = cv2.dilate(logo, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 3)), iterations=1)
    logo = cv2.bitwise_and(logo, search)
    return logo, kept


def _fill_internal_gaps(mask):
    """Close small breaks and fill holes inside mask components."""
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                                       CONTROL_HOLE_CLOSE_KERNEL))
    inv = cv2.bitwise_not(closed)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(inv, connectivity=8)
    holes = np.zeros_like(mask)
    h, w = mask.shape[:2]
    for label in range(1, n):
        x, y, bw, bh, _ = stats[label]
        touches_border = x == 0 or y == 0 or x + bw >= w or y + bh >= h
        if not touches_border:
            holes[labels == label] = 255
    return cv2.bitwise_or(closed, holes)


def _build_controlled_halo_mask(frame_rgb, arch_bot, keel_cx_at, keel_y0, keel_y1):
    """Build a deliberate Halo mask from geometry instead of dark-pixel flooding."""
    h, w = frame_rgb.shape[:2]
    row_idx = np.arange(h)[:, np.newaxis]

    # arch_bot is the lower Halo edge; fill upward to the video border when requested.
    if ARCH_BODY_TO_TOP:
        arch_top = np.zeros_like(arch_bot[np.newaxis, :])
    else:
        arch_top = np.maximum(0, arch_bot[np.newaxis, :] - ARCH_BODY_ABOVE)
    arch_bottom = np.minimum(h - 1, arch_bot[np.newaxis, :] + ARCH_BODY_BELOW)
    arch_mask = ((row_idx >= arch_top) & (row_idx <= arch_bottom)).astype(np.uint8) * 255
    arch_mask = cv2.morphologyEx(arch_mask, cv2.MORPH_CLOSE,
                                 cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                                          ARCH_BODY_CLOSE_KERNEL))

    keel_mask = np.zeros((h, w), dtype=np.uint8)
    keel_h = max(keel_y1 - keel_y0, 1)
    for row_y in range(max(0, keel_y0), min(h, keel_y1)):
        t = (row_y - keel_y0) / keel_h
        hw = int(round(CONTROL_KEEL_TOP_HALF_WIDTH * (1.0 - t) +
                       CONTROL_KEEL_BOTTOM_HALF_WIDTH * t))
        if row_y - keel_y0 < CONTROL_KEEL_JUNCTION_ROWS:
            hw += CONTROL_KEEL_JUNCTION_BONUS
        cx = keel_cx_at(row_y)
        keel_mask[row_y, max(0, cx - hw):min(w, cx + hw + 1)] = 255

    yoke_rows = min(CONTROL_YOKE_ROWS, max(keel_y1 - keel_y0, 1))
    for dy in range(yoke_rows):
        row_y = keel_y0 + dy
        if row_y < 0 or row_y >= h:
            continue
        t = dy / max(yoke_rows - 1, 1)
        hw = int(round(CONTROL_YOKE_TOP_HALF_WIDTH * (1.0 - t) +
                       CONTROL_YOKE_BOTTOM_HALF_WIDTH * t))
        cx = keel_cx_at(row_y)
        keel_mask[row_y, max(0, cx - hw):min(w, cx + hw + 1)] = 255

    # The arch/keel convergence is rounded, not a hard triangular join.
    junction_y = int(np.clip(keel_y0 + CONTROL_JUNCTION_ELLIPSE_DY, 0, h - 1))
    junction_cx = int(np.clip(keel_cx_at(junction_y), 0, w - 1))
    cv2.ellipse(keel_mask,
                (junction_cx, junction_y),
                (CONTROL_JUNCTION_ELLIPSE_X, CONTROL_JUNCTION_ELLIPSE_Y),
                0, 0, 360, 255, thickness=cv2.FILLED)

    logo_search = np.ones((h, w), dtype=np.uint8) * 255
    logo_mask, logo_components = _extract_f1_logo_mask(frame_rgb, logo_search)

    combined = cv2.bitwise_or(cv2.bitwise_or(arch_mask, keel_mask), logo_mask)
    combined = _fill_internal_gaps(combined)
    combined = cv2.morphologyEx(combined, cv2.MORPH_CLOSE,
                               cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                                        CONTROL_FINAL_CLOSE_KERNEL))
    combined = cv2.dilate(combined,
                          cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                                   CONTROL_FINAL_DILATE_KERNEL),
                          iterations=CONTROL_FINAL_DILATE_ITER)
    return combined, logo_components



def build_frame_mask(frame_rgb,
                     prev_keel_cx=None,
                     return_debug=False):
    h, w = frame_rgb.shape[:2]
    blurred = cv2.GaussianBlur(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY),
                               (5, 5), 0).astype(np.float32)
    gray_u8 = np.clip(blurred, 0, 255).astype(np.uint8)
    band_h = int(ARCH_Y_MAX * h)

    # Neutralize the F1 logo before the arch edge detector sees it.
    arch_input = blurred[:band_h].copy()
    sky_right = blurred[5:30, LOGO_W_PX + 20:min(LOGO_W_PX + 180, w)]
    sky_val = float(np.median(sky_right)) if sky_right.size > 0 else 160.0
    arch_input[:min(LOGO_H_PX, band_h), :LOGO_W_PX] = sky_val

    # Sobel-Y estimates the lower arch edge; black contours are filled after keel fit.
    sobel = cv2.Sobel(arch_input, cv2.CV_32F, 0, 1, ksize=5)
    sobel = np.maximum(sobel, 0)
    sobel[:4, :] = 0
    sobel_sm = cv2.GaussianBlur(sobel, (41, 1), 0)
    sobel_sm[:min(LOGO_H_PX + 12, band_h), :LOGO_W_PX + 8] = 0

    arch_bot = np.argmax(sobel_sm, axis=0).astype(np.float32)
    strength = sobel_sm.max(axis=0)
    arch_bot[strength < ARCH_WEAK_EDGE_FRAC * max(float(strength.max()), 1.0)] = band_h
    from scipy.ndimage import median_filter as _mf
    arch_bot = _mf(arch_bot, size=25)
    arch_bot = cv2.GaussianBlur(arch_bot.reshape(1, -1), (81, 1), 0).ravel()
    arch_bot = np.clip(arch_bot + ARCH_CONTOUR_PAD, 0, band_h)

    # Do not paint a top slab here. The arch edge only bounds the black contour search.

    # Keel probes need the right side of the frame; the old 0.40-0.60 window clipped it.
    sx0_keel = int(KEEL_SEARCH_X0 * w)
    sx1_keel = int(KEEL_SEARCH_X1 * w)
    probe_pts = []

    for y_p in KEEL_PROBE_YS:
        y0p = max(y_p - KEEL_PROBE_HALF_H, 0)
        y1p = min(y_p + KEEL_PROBE_HALF_H + 1, h)
        if y0p >= y1p:
            continue

        roi_p = blurred[y0p:y1p, sx0_keel:sx1_keel]
        if roi_p.size == 0:
            continue

        col = roi_p.mean(axis=0)
        col_sm = cv2.GaussianBlur(col.reshape(1, -1), (15, 1), 0).ravel()
        med = float(np.median(col_sm))
        cx_loc = int(np.argmin(col_sm))
        contrast = med - float(col_sm[cx_loc])

        if cx_loc < 12 or cx_loc > len(col_sm) - 13 or contrast < KEEL_MIN_CONTRAST:
            continue

        dark = col_sm < med - max(12.0, 0.40 * contrast)
        left_edge = cx_loc
        while left_edge > 0 and dark[left_edge - 1]:
            left_edge -= 1
        right_edge = cx_loc
        while right_edge < len(dark) - 1 and dark[right_edge + 1]:
            right_edge += 1

        if right_edge - left_edge + 1 > KEEL_MAX_VALLEY_WIDTH:
            continue

        probe_pts.append((float(y_p), sx0_keel + 0.5 * (left_edge + right_edge)))

    used_fallback = False
    if len(probe_pts) >= 2:
        ys_arr = np.array([p[0] for p in probe_pts], dtype=np.float64)
        cxs_arr = np.array([p[1] for p in probe_pts], dtype=np.float64)
        slope, intercept, keep = _fit_keel_line(ys_arr, cxs_arr)
        inlier_count = int(keep.sum())
    else:
        slope = 0.0
        intercept = float(prev_keel_cx) if prev_keel_cx is not None else 0.62 * w
        inlier_count = 0
        used_fallback = True

    def keel_cx_at(y):
        return int(np.clip(slope * y + intercept, sx0_keel, sx1_keel))

    keel_mask = np.zeros((h, w), dtype=np.uint8)
    top_cx = keel_cx_at(max(KEEL_PROBE_YS[0], band_h))
    lo = max(0, top_cx - 24)
    hi = min(w, top_cx + 25)
    keel_y0 = max(0, int(np.median(arch_bot[lo:hi])) - 16)
    keel_y1 = int(KEEL_Y1 * h)
    keel_h = max(keel_y1 - keel_y0, 1)

    for row_y in range(keel_y0, keel_y1):
        t = (row_y - keel_y0) / keel_h
        hw = int(round(KEEL_TOP_HALF_WIDTH * (1.0 - t) + KEEL_BOTTOM_HALF_WIDTH * t))
        if row_y - keel_y0 < KEEL_JUNCTION_ROWS:
            hw += KEEL_JUNCTION_BONUS
        cx = keel_cx_at(row_y)
        keel_mask[row_y, max(0, cx - hw):min(w, cx + hw + 1)] = 255

    # The arch-keel fork has rounded Y shoulders below the Sobel arch contour.
    # Add a local yoke patch so those curved inner edges do not survive inpaint.
    yoke_rows = min(KEEL_YOKE_ROWS, max(keel_y1 - keel_y0, 1))
    for dy in range(yoke_rows):
        row_y = keel_y0 + dy
        t = dy / max(yoke_rows - 1, 1)
        hw = int(round(KEEL_YOKE_TOP_HALF_WIDTH * (1.0 - t) + KEEL_YOKE_BOTTOM_HALF_WIDTH * t))
        cx = keel_cx_at(row_y)
        keel_mask[row_y, max(0, cx - hw):min(w, cx + hw + 1)] = 255

    combined, logo_components = _build_controlled_halo_mask(
        frame_rgb, arch_bot, keel_cx_at, keel_y0, keel_y1)
    keel_cx_out = keel_cx_at(int(np.mean(KEEL_PROBE_YS)))

    if return_debug:
        return combined, keel_cx_out, {
            'probe_count': len(probe_pts),
            'inlier_count': inlier_count,
            'used_fallback': used_fallback,
            'slope': float(slope),
            'keel_cx': int(keel_cx_out),
            'logo_components': int(logo_components),
            'mask_area': int((combined > 0).sum()),
        }
    return combined, keel_cx_out


# Build masks, threading keel_cx forward as the no-probe fallback.
masks = []
keel_debug = []
prev_keel = None
for f in frames:
    m, prev_keel, dbg = build_frame_mask(f, prev_keel_cx=prev_keel, return_debug=True)
    masks.append(m)
    keel_debug.append(dbg)

probe_counts = np.array([d['probe_count'] for d in keel_debug])
mask_areas = np.array([d['mask_area'] for d in keel_debug], dtype=np.float64)
keel_cxs = np.array([d['keel_cx'] for d in keel_debug], dtype=np.float64)
logo_components = np.array([d['logo_components'] for d in keel_debug])
print('Per-frame masks built:', len(masks))
print('Keel probes: min', int(probe_counts.min()), 'median', int(np.median(probe_counts)),
      'fallback frames', int(sum(d['used_fallback'] for d in keel_debug)))
print('Keel cx: min', int(keel_cxs.min()), 'max', int(keel_cxs.max()),
      'median', round(float(np.median(keel_cxs)), 1))
print('Logo contours: min', int(logo_components.min()),
      'median', int(np.median(logo_components)), 'max', int(logo_components.max()))
print('Mask area variation:', round(float(mask_areas.std() / mask_areas.mean()), 3))

picks4 = np.linspace(0, len(frames) - 1, 4, dtype=int)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for col, i in enumerate(picks4):
    axes[col].imshow(overlay_mask(frames[i], masks[i]))
    axes[col].set_title(f'mask [{i}]'); axes[col].axis('off')
plt.suptitle('Masks - filled arch body + fitted keel + F1 logo contour', fontsize=11)
plt.tight_layout(); plt.show()


In [ ]:
# masks is already built per-frame in the detection cell above.
# warped_masks alias kept so the drift visualisation cell still runs.
warped_masks = masks
print('masks ready:', len(masks))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for col, i in enumerate(picks4):
    axes[col].imshow(overlay_mask(frames[i], masks[i]))
    axes[col].set_title(f'frame {i}'); axes[col].axis('off')
plt.suptitle('Per-frame Halo masks — filled arch body + fitted keel',
             fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# Measure centroid drift of the mask over time
def mask_centroid(m):
    ys, xs = np.where(m > 0)
    if len(xs) == 0:
        return (np.nan, np.nan)
    return (float(xs.mean()), float(ys.mean()))

centroids = [mask_centroid(m) for m in warped_masks]
cx0, cy0  = centroids[0]
drifts    = [float(np.hypot(cx - cx0, cy - cy0)) for cx, cy in centroids]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(drifts, lw=2, color='steelblue')
ax.axhline(5, ls='--', color='gray', lw=1, label='5 px (roll-hoop cam threshold)')
ax.set_xlabel('Frame'); ax.set_ylabel('Centroid drift (px)')
ax.set_title('Halo mask drift vs. frame 0 '
             '(visor cam: expect significant drift as the driver looks around)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

max_drift = max(drifts)
print('Max drift:', round(max_drift, 1), 'px    Mean:', round(float(np.mean(drifts)), 1), 'px')

# For a visor cam the Halo almost always moves — use warped masks by default
USE_STATIC = max_drift < 5.0
if USE_STATIC:
    masks = [MASK.copy() for _ in frames]
    print('-> Drift negligible: using STATIC mask (unexpected for a visor cam — double-check clip)')
else:
    masks = warped_masks
    print('-> Drift detected: using per-frame WARPED masks (expected for visor cam)')

In [ ]:
# ── Accelerator detection ─────────────────────────────────────────────────────
# Priority: CUDA (NVIDIA) > MPS (Apple Silicon via PyTorch) > CPU
# MLX (Apple's native ML framework) is detected and reported but cannot be used
# here because RAFT and LaMa are PyTorch models — MLX would require porting them.
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    _accel = f'CUDA — {torch.cuda.get_device_name(0)}'
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    _accel = 'MPS (Apple Silicon)'
else:
    DEVICE = torch.device('cpu')
    _accel = 'CPU'

# Report MLX separately — it exists alongside PyTorch on Apple Silicon but
# cannot accelerate these specific models without a full port.
try:
    import mlx.core as mx
    _mlx_info = f'MLX {mx.__version__} detected (default device: {mx.default_device()}) — not used (PyTorch models only)'
except ImportError:
    _mlx_info = 'MLX not installed'

print(f'PyTorch device : {DEVICE}  ({_accel})')
print(f'MLX status     : {_mlx_info}')

# ── RAFT-small — used by Method 2 (temporal propagation) ─────────────────────
_raft_w         = Raft_Small_Weights.DEFAULT
raft_model      = raft_small(weights=_raft_w).eval().to(DEVICE)
raft_transforms = _raft_w.transforms()

print(f'RAFT-small     : {sum(p.numel() for p in raft_model.parameters()):,} params  →  ready on {DEVICE}')


## 5. Method 1 — LaMa Large-Mask Inpainting

**Why VGG PatchMatch failed here:**

PatchMatch copies patches from the *same frame's* unmasked pixels. For the arch
(full-width top-24% mask), the only candidate source patches are from the cockpit,
steering wheel, and track below — not sky. The model has no way to hallucinate
coherent sky content, so it pastes in wrong-colored blocks and produces visible
artifacts.

**LaMa (Large Mask inpainting, Suvorov et al., WACV 2022):**

LaMa is a convolutional network trained end-to-end on millions of natural scene
images specifically to fill large masked regions with perceptually coherent content.
It uses **Fast Fourier Convolutions** (FFC) whose receptive field covers the entire
image at every layer, so it can reason about scene structure globally — sky goes
above the horizon, track markings continue across the gap — rather than copying
local texture. This is what commercial tools (Photoshop Generative Fill, etc.) use.

We run LaMa inference with pretrained weights (no training on this video needed)
to fill both the arch and keel mask per frame, then apply the soft boundary blend.


In [ ]:
from simple_lama_inpainting import SimpleLama
from PIL import Image as _PIL_Image

# Runs on whichever accelerator DEVICE was set to in cell-neural-setup.
# CPU: ~3-5 s/frame → 25 min for 300 frames.  MPS/CUDA: ~0.3-0.5 s/frame → ~2 min.
lama = SimpleLama(device=DEVICE)
print('LaMa model ready on', DEVICE)


def lama_inpaint(frame_rgb, mask):
    """LaMa large-mask inpainting for a single frame.

    Converts to PIL, runs LaMa inference, soft-blends the result at mask edges
    so the filled region merges smoothly with original content.
    """
    img_pil    = _PIL_Image.fromarray(frame_rgb)
    mask_pil   = _PIL_Image.fromarray(mask)
    result_pil = lama(img_pil, mask_pil)
    result     = np.array(result_pil)

    # Soft boundary blend — keep original pixels outside the mask intact
    soft = cv2.GaussianBlur(mask.astype(np.float32) / 255.0, (7, 7), 0)[:, :, np.newaxis]
    return (result.astype(np.float32) * soft +
            frame_rgb.astype(np.float32) * (1.0 - soft)).astype(np.uint8)


inpainted_spatial = []
for i, (f, m) in enumerate(zip(frames, masks)):
    inpainted_spatial.append(lama_inpaint(f, m))
    if (i + 1) % 30 == 0 or i == 0:
        print(f'  {i+1}/{len(frames)} frames done')
print('LaMa inpainting done.')

picks4 = np.linspace(0, len(frames) - 1, 4, dtype=int)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for col, i in enumerate(picks4):
    axes[0, col].imshow(frames[i])
    axes[0, col].set_title(f'Original [{i}]'); axes[0, col].axis('off')
    axes[1, col].imshow(inpainted_spatial[i])
    axes[1, col].set_title(f'LaMa [{i}]'); axes[1, col].axis('off')
plt.suptitle('Method 1 — LaMa Large-Mask Inpainting', fontsize=13)
plt.tight_layout(); plt.show()


## 6. Method 2 — RAFT Optical Flow Temporal Propagation

**Algorithm:**

For each frame $t$:
1. Compute dense optical flow $f: \text{frame}_{t-1}^{\text{clean}} \to \text{frame}_t^{\text{VGG}}$
   using **RAFT** (Recurrent All-Pairs Field Transforms, Teed & Deng 2020).
   RAFT encodes both frames with a shared CNN feature extractor, builds an
   all-pairs correlation volume across feature positions, and iteratively
   refines the flow field with a convolutional GRU.
2. **Backward-warp** the previous clean frame using the estimated flow.
3. **Distance-transform alpha blend** — pixels deep inside the mask (far from
   the boundary) are taken from the warp; pixels near the boundary blend back
   to the current frame to avoid boundary ghosts.

**Why RAFT over Farneback:**
Farneback fits local polynomial expansions — it breaks down under the fast
lateral camera motion at Suzuka and the layered backgrounds (ferris wheel,
grandstands at speed).  RAFT's learned feature representations handle large
displacements, motion blur, and occlusion far more robustly, giving temporally
stable fills that don't shimmer frame-to-frame.

In [ ]:
def neural_temporal_inpaint(prev_clean, curr_frame, mask, curr_neural):
    """RAFT Optical Flow Temporal Propagation.

    Flow estimated between prev_clean and curr_neural (VGG-inpainted current
    frame) to measure background motion without halo contamination.
    The previous clean frame is then backward-warped into the current mask.
    """
    # ── RAFT optical flow ────────────────────────────────────────────────────
    t1 = torch.from_numpy(prev_clean).permute(2, 0, 1).unsqueeze(0)   # uint8
    t2 = torch.from_numpy(curr_neural).permute(2, 0, 1).unsqueeze(0)
    t1t, t2t = raft_transforms(t1, t2)
    with torch.no_grad():
        flows = raft_model(t1t.to(DEVICE), t2t.to(DEVICE))
    flow = flows[-1].squeeze(0).cpu().numpy()   # (2, H, W)  [dx, dy]

    # ── Backward warp ────────────────────────────────────────────────────────
    h, w    = curr_frame.shape[:2]
    yg, xg  = np.mgrid[0:h, 0:w].astype(np.float32)
    map_x   = xg - flow[0]
    map_y   = yg - flow[1]
    warped  = cv2.remap(prev_clean.astype(np.float32), map_x, map_y,
                        cv2.INTER_LINEAR,
                        borderMode=cv2.BORDER_REPLICATE).astype(np.uint8)

    # ── Distance-transform alpha blend ───────────────────────────────────────
    dist  = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    alpha = np.clip(dist / 8.0, 0.0, 1.0)[:, :, np.newaxis]
    return (warped.astype(np.float32) * alpha +
            curr_frame.astype(np.float32) * (1.0 - alpha)).astype(np.uint8)

In [ ]:
RESET_INTERVAL = 60   # reset temporal chain every N frames to prevent error accumulation

inpainted_temporal = [inpainted_spatial[0]]

for i in range(1, len(frames)):
    if i % RESET_INTERVAL == 0:
        prev = inpainted_spatial[i - 1]
    else:
        prev = inpainted_temporal[-1]
    filled = neural_temporal_inpaint(prev, frames[i], masks[i], inpainted_spatial[i])
    inpainted_temporal.append(filled)
    if (i + 1) % 30 == 0:
        print(f'  {i+1}/{len(frames)} frames done')

print('RAFT temporal inpainting done.')

fig, axes = plt.subplots(3, 4, figsize=(16, 11))
for col, i in enumerate(picks4):
    axes[0, col].imshow(frames[i])
    axes[0, col].set_title(f'Original [{i}]');      axes[0, col].axis('off')
    axes[1, col].imshow(inpainted_spatial[i])
    axes[1, col].set_title(f'LaMa Spatial [{i}]');  axes[1, col].axis('off')
    axes[2, col].imshow(inpainted_temporal[i])
    axes[2, col].set_title(f'RAFT Temporal [{i}]'); axes[2, col].axis('off')

plt.suptitle('Original  /  Method 1: LaMa Inpainting  /  Method 2: RAFT Temporal Flow',
             fontsize=13)
plt.tight_layout(); plt.show()


## 7. Output

Side-by-side playback animation and exported MP4 files.

In [ ]:
plt.rcParams['animation.embed_limit'] = 50  # MB

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=80)
for ax, title in zip(axes, ['Original', 'LaMa Spatial', 'RAFT Temporal']):
    ax.set_title(title, fontsize=11); ax.axis('off')

ims = [
    axes[0].imshow(frames[0]),
    axes[1].imshow(inpainted_spatial[0]),
    axes[2].imshow(inpainted_temporal[0]),
]

def update(i):
    ims[0].set_data(frames[i])
    ims[1].set_data(inpainted_spatial[i])
    ims[2].set_data(inpainted_temporal[i])
    return ims

ani = animation.FuncAnimation(fig, update, frames=len(frames),
                               interval=1000.0 / FPS, blit=True)
plt.close()
HTML(ani.to_jshtml())


In [ ]:
OUT = Path('output')
OUT.mkdir(exist_ok=True)

def write_mp4(path, frame_list, fps):
    h, w = frame_list[0].shape[:2]
    vw   = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for f in frame_list:
        vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
    vw.release()
    print(' ', path.name, ' ', len(frame_list), 'frames,',
          round(path.stat().st_size / 1e6, 1), 'MB')

print('Writing videos...')
write_mp4(OUT / 'original.mp4',  list(frames),         FPS)
write_mp4(OUT / 'spatial.mp4',   inpainted_spatial,    FPS)
write_mp4(OUT / 'temporal.mp4',  inpainted_temporal,   FPS)
print('Done. Files in', str(OUT))